# 01. Data Cleaning, Target Formulation & Leakage Prevention

## 1. Problem Formulation: Credit Default Underwriting
In retail banking and peer-to-peer (P2P) lending, an underwriting engine must answer one foundational question at the moment an applicant submits their loan request:
> **"Given the verifiable borrower attributes and credit bureau history available today, what is the probability that this applicant will default (charge off) before the loan term expires?"**

### Target Definition Strategy:
- **`Class 0: Fully Paid`** (Good Borrower) -> Principal and accrued interest returned in full.
- **`Class 1: Charged Off / Default`** (Bad Borrower) -> Severely delinquent, payments ceased, debt written off as uncollectible.
- **`Drop: Current / In-Progress Loans`** -> Active loans have unobserved terminal outcomes (**right-censoring**). Labeling them as 0 or 1 introduces severe label contamination.

In [1]:
import sys
from pathlib import Path

# Ensure project root is in Python module path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.data_prep import (
    load_raw_data,
    filter_closed_loans_and_define_target,
    clean_loan_data,
    inspect_data_health,
    LEAKAGE_COLUMNS_REASONS
)

print("Environment initialized successfully.")

Environment initialized successfully.


## 2. Load Raw Lending Club Data
We load the raw data stored in `data/raw/loan.csv`.

In [2]:
raw_data_path = PROJECT_ROOT / "data" / "raw" / "loan.csv"
df_raw = load_raw_data(raw_data_path)
print(f"Raw dataset dimensions: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

Loading raw loan data from: loan.csv...


Loaded raw records: 39,717 rows x 111 columns.
Raw dataset dimensions: 39,717 rows x 111 columns


## 3. Filter Terminal Loans and Define Binary Target
We drop ongoing `Current` loans (right-censored) and create the binary `target` column.

In [3]:
df_closed, target_summary = filter_closed_loans_and_define_target(df_raw)
print(f"Closed loans dataset: {df_closed.shape[0]:,} rows x {df_closed.shape[1]} columns")

Closed loans dataset: 38,577 rows x 112 columns


## 4. Target Leakage: Identifying and Removing Post-Issuance Features

### What is Target Leakage in Credit Risk?
**Data Leakage** occurs when a predictive model is trained on features that would **never be available at the time of prediction in production**.

In credit scoring, loan underwriting happens *before* money is disbursed. Any column populated or updated *after* the loan is issued is a post-issuance feature. If retained, the model will learn trivial shortcut rules (e.g. "if `recoveries` > 0, borrower defaulted"), resulting in near-100% training accuracy but complete failure in real-world deployment.

Here is the explicit audit of every leakage column removed:

In [4]:
leakage_df = pd.DataFrame([
    {"Feature": col, "Leakage Mechanism": reason}
    for col, reason in LEAKAGE_COLUMNS_REASONS.items()
])
display(leakage_df)

,Feature,Leakage Mechanism
0,funded_amnt,Post-origination: Final amount funded by inves...
1,funded_amnt_inv,Post-origination: Portion of funded amount com...
2,out_prncp,Post-issuance payment status: Remaining outsta...
3,out_prncp_inv,Post-issuance payment status: Remaining outsta...
4,total_pymnt,Post-issuance target leak: Total payments rece...
5,total_pymnt_inv,Post-issuance target leak: Total payments rece...
6,total_rec_prncp,Post-issuance target leak: Principal received ...
7,total_rec_int,Post-issuance target leak: Total interest rece...
8,total_rec_late_fee,Post-issuance target leak: Late fees collected...
9,recoveries,Direct target leak: Gross post-charge-off reco...


## 5. Data Cleaning, Type Parsing, and Missing Value Handling

### Data Type Conversions:
1. **`term`**: String with values like `' 36 months'` or `' 60 months'`. Stripped to integers `36` and `60`.
2. **`int_rate`**: String percentage (e.g. `'10.65%'`). Parsed to numeric float (`10.65`).
3. **`revol_util`**: String percentage (e.g. `'83.70%'`). Parsed to numeric float (`83.70`).
4. **`emp_length`**: String with ordinal categories (`'< 1 year'`, `'1 year'`, ..., `'10+ years'`). Parsed to numeric years (`0` to `10`).

### Justified Missing Value Strategies:
1. **`pub_rec_bankruptcies` (697 missing, 1.81%)**: Imputed with `0.0`. In credit bureau reports, absence of public record bankruptcies indicates clean record (confirmed: 95.8% of population has 0 bankruptcies).
2. **`revol_util` (50 missing, 0.13%)**: Imputed with median (~49.3%). Extremely low missingness; median preserves distribution without dropping records.
3. **`emp_length_years` (1,033 missing, 2.68%)**: Imputed with median (`4.0` years) and flagged with a binary indicator `emp_length_missing` (`1` if missing, `0` otherwise) to allow tree models to capture non-random missing patterns.

Now we execute `clean_loan_data` and save to `data/processed/`.

In [5]:
processed_dir = PROJECT_ROOT / "data" / "processed"
df_clean, clean_summary = clean_loan_data(
    df_closed,
    target_col="target",
    missing_thresh=0.50,
    save_processed_dir=processed_dir
)

print(f"\n--- CLEANING AUDIT SUMMARY ---")
print(f"Initial raw rows:              {clean_summary['initial_shape'][0]:,}")
print(f"Initial columns:               {clean_summary['initial_shape'][1]}")
print(f"High-missing (>50%) dropped:   {clean_summary['high_missing_dropped_count']}")
print(f"Leakage features dropped:      {clean_summary['leakage_dropped_count']}")
print(f"Admin/Noise features dropped:  {clean_summary['admin_noise_dropped_count']}")
print(f"Total columns dropped:         {clean_summary['total_columns_dropped']}")
print(f"Duplicates removed:            {clean_summary['duplicates_removed']}")
print(f"Final clean dataset shape:     {clean_summary['final_shape']}")

Saved cleaned dataset to:
  - C:\Users\dr_ra\Desktop\loan-Prediction\data\processed\loans_cleaned.parquet
  - C:\Users\dr_ra\Desktop\loan-Prediction\data\processed\loans_cleaned.csv

--- CLEANING AUDIT SUMMARY ---
Initial raw rows:              38,577
Initial columns:               112
High-missing (>50%) dropped:   57
Leakage features dropped:      15
Admin/Noise features dropped:  16
Total columns dropped:         88
Duplicates removed:            0
Final clean dataset shape:     (38577, 25)


## 6. Verify Cleaned Dataset Health
We confirm that the processed dataset in `data/processed/loans_cleaned.parquet` has zero missing values and correct data types.

In [6]:
health_clean = inspect_data_health(df_clean, target_col="target")
print("Remaining missing cells:", health_clean["total_missing_cells"])
print("Memory usage (MB):      ", health_clean["memory_usage_mb"])
print("Retained feature dtypes :", health_clean["dtype_counts"])
print("\nRetained Features:", clean_summary["retained_features"])

Remaining missing cells: 0
Memory usage (MB):       23.1
Retained feature dtypes : {'object': 8, 'int64': 7, 'float64': 7, 'int32': 3}

Retained Features: ['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'pub_rec_bankruptcies', 'emp_length_years', 'emp_length_missing']
